# PyTorch: tensors, autograd and a training loop

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Use tensors and autograd
- Train a small network with a DataLoader
- Compare learning rates

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

## 1. Autograd

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w * 3 - 9) ** 2
loss.backward()
print(w.grad)   # d/dw (3w − 9)² = 6(3w − 9) = −18 at w = 2

## 2. Data: y = sin(x) with noise

In [ ]:
X = torch.linspace(-3, 3, 2000).unsqueeze(1)
y = torch.sin(X) + 0.1 * torch.randn_like(X)
perm = torch.randperm(len(X)); tr, va = perm[:1600], perm[1600:]
train_dl = DataLoader(TensorDataset(X[tr], y[tr]), batch_size=64, shuffle=True)
val_dl = DataLoader(TensorDataset(X[va], y[va]), batch_size=256)

## 3. Training loop

In [ ]:
def train(lr, epochs=40):
    model = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = nn.functional.mse_loss(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return sum(nn.functional.mse_loss(model(xb.to(device)), yb.to(device)).item() for xb, yb in val_dl) / len(val_dl)
for lr in [1e-4, 1e-3, 1e-2, 1e-1]:
    print(f'lr={lr:g} val MSE={train(lr):.4f}')

## Exercises
1. Remove `opt.zero_grad()` and see what happens to the loss.
2. Add early stopping: stop when validation loss hasn't improved for 5 epochs.

In [ ]:
# your work here

## Solutions
Without `zero_grad`, gradients accumulate across batches, so steps grow and training becomes unstable. For early stopping, track the best validation loss each epoch and `break` after 5 epochs without improvement.